# Day 9 — Structure-Aware and Semantic Chunking Strategies
---
> **Phase:** 2 — RAG Systems  
> **Status:** ✅ Complete

## 🧠 Key Concepts

### 1. Why Fixed Chunking Fails in Production

Day 8 used word-level fixed chunking. The known limitation:

**Problem 1 — Topic bleeding:**
A 100-word window starting halfway through Section 5 and ending halfway through Section 6 produces a chunk containing content from two unrelated topics. The LLM receives this as a single unit of evidence and synthesizes the two concepts into a coherent-sounding but fabricated answer.

Example:
```
Section 5: Late payments incur a 2% monthly penalty.
Section 6: Any disputes must be raised within 14 days.
```
Fixed chunker produces: chunk containing both → LLM may hallucinate:
"A 2% penalty must be disputed within 14 days."
This is **confidently wrong** — the most dangerous failure mode in production RAG.

**Problem 2 — Loss of context:**
A chunk starting mid-sentence loses the subject of that sentence. The LLM sees "2% monthly penalty" without knowing what it applies to.

**Root cause:** Fixed chunking ignores document structure entirely.

### 2. The Three Families of Chunking Strategies

```
Family 1 — Fixed chunking:       splits by word count
Family 2 — Structure-aware:      splits by visible structure signals
Family 3 — Semantic chunking:    splits by meaning shift
```

**Fixed chunking** — last resort only. Use as a baseline when nothing else is available.

**Structure-aware chunking** — best for documents with visible structure:
- Legal HR policy PDFs with numbered sections
- Technical documentation with headers
- Job descriptions with labeled sections
- Any document where a human reader can visually identify section boundaries

**Semantic chunking** — best for documents with no visible structure:
- Scraped web content with no headers
- Long prose documents
- Content-heavy paragraphs where topic shifts happen mid-paragraph

**Decision rule:**
```
Does the document have visible structure signals?
  → Yes: Structure-aware chunking

Does it have NO visible structure but coherent topics?
  → Yes: Semantic chunking

Is it truly unstructured and you need a quick baseline?
  → Fixed chunking as last resort only
```

### 3. Natural Unit — The Core Concept

Before choosing a chunking strategy, ask: **what is the natural unit of meaning in this document?**

| Document Type | Natural Unit | Never Split Across |
|--------------|-------------|-------------------|
| HR Policy PDF | Numbered section + its body | Section header and body |
| Research paper | Paragraph | Mid-paragraph |
| Support transcript | Full user conversation thread | Related topic exchange |
| Legal contract | Clause | Clause boundary |
| Job description | Labeled section | Section content |

For support transcripts with multiple topics per conversation:
- Chunk by topic entity (e.g. order number) not by user or by message
- One order inquiry = one chunk, regardless of how many turns it spans

### 4. Structure-Aware Chunking — Algorithm

**Split signals for HR policy documents:**
```
Signal 1 — \n\n        → paragraph boundary
Signal 2 — ^\d+\.      → numbered section header (regex)
```

**Why regex over simple string methods:**
- Need to match a pattern, not a fixed string
- Pattern: start of string + one or more digits + literal period + space
- Regex: `^\d+\. `

**Algorithm:**
```
Step 1: Split text on \n\n → list of paragraphs
Step 2: Iterate through paragraphs
Step 3: If paragraph matches ^\d+\. → new section boundary
         → finalize current_chunk → save to chunks → reset current_chunk
         → start fresh with this header
Step 4: If paragraph is body text → append to current_chunk
Step 5: After loop → finalize whatever remains in current_chunk
Step 6: Filter out title-only chunks (len < 50)
```

**Key pattern — two lists:**
```python
chunks = []        # final result — all completed chunks
current_chunk = [] # temporary — pieces being built right now
```

**The finalization moment:**
Finalization happens in exactly TWO places:
1. When you hit a new section header (save previous section)
2. After the loop ends (save the last section)
Never inside the loop on every iteration.

### 5. Semantic Chunking — Algorithm

**Core insight:** Similar sentences produce vectors close together in embedding space. Different topic sentences produce vectors far apart. A drop in cosine similarity between consecutive sentences signals a topic boundary.

**Algorithm:**
```
Step 1: Split document into sentences (use nltk.sent_tokenize)
Step 2: Filter out fragments (len < 30)
Step 3: Embed each sentence → matrix of shape (n_sentences, 384)
Step 4: Compute cosine similarity between each consecutive pair
        sim(S0,S1), sim(S1,S2), ..., sim(Sn-2, Sn-1)
        → n-1 similarity scores for n sentences
Step 5: Where similarity drops below threshold → insert split point
Step 6: Group sentences between split points into a chunk
```

**Why nltk.sent_tokenize over splitting on '.':**
- Splitting on `.` treats `1.` in numbered lists as sentence endings
- Treats abbreviations like `Dr.`, `Mr.` as sentence endings
- Treats decimal numbers like `2.5` as sentence endings
- `nltk.sent_tokenize` handles all these edge cases automatically

**Threshold tuning:**
```
Higher threshold → more pairs fall below it → more splits → smaller chunks
Lower threshold  → fewer pairs fall below it → fewer splits → larger chunks
Too high → fragments (single sentence chunks)
Too low  → no splits (entire document as one chunk)
Solution → tune empirically by testing on your actual data
```

**Consecutive pairs math:**
```
9 sentences → 8 consecutive pairs → always n-1 pairs for n sentences
```

### 6. Production Routing — Choosing the Right Strategy

**The chunking router pattern:**
```
Document comes in
      ↓
Inspect the text
      ↓
Has numbered sections / headers?  → Structure-aware
Has paragraphs but no headers?    → Semantic
Is a transcript / conversation?   → Speaker-turn or topic-entity based
Pure unstructured text?           → Semantic as fallback
```

**LLM-guided document routing:**
Pass first 500-1000 tokens to LLM → classify document type and structure → router selects chunking strategy.
Uses structured JSON output pattern from Day 3.

**Limitation of sampling first 1000 tokens:**
Document structure may change mid-document. Two solutions:
1. Sample more tokens → better coverage, higher LLM cost
2. Adaptive chunking → inspect structure signal by signal during chunking pass itself, switch strategy mid-document if needed

**Adaptive chunking** is the more robust production approach. Still an open research problem — no single perfect solution exists yet.

### 7. Hierarchical Chunking

A single document can have mixed structure. The production solution:

```
Level 1 — Structure-aware:  split on numbered sections / headers
Level 2 — Semantic:         split long passages within each section
                             if they exceed a length threshold
```

Best for: legal contracts, HR policies, technical manuals — any document with both macro structure and long prose within sections.

This is what production RAG systems use for complex documents.

### 8. Limitations of Semantic Chunking on Structured Documents

When `nltk.sent_tokenize` processes a structured document:
- Section headers get joined with the first sentence of their body: `"Attendance, Working Hours\n\nEmployees are required..."`
- This creates mixed vectors — the embedding contains both header topic words and body content words
- Mixed vectors confuse the similarity scores → incorrect split points
- Result: orphaned sentences, incorrect groupings

**Practical lesson:** Always look at your actual data before deciding on chunking strategy. Semantic chunking works best on clean prose without structural noise mixed in.

### 9. Title Chunk Filtering

Structure-aware chunking on a document with a title produces a title-only chunk:
```
Chunk 1: HUMAN RESOURCES POLICY   ← 22 characters, zero information
```

**Why this is a production problem:**
The retriever computes cosine similarity against all chunks. A title chunk may score moderately high for many queries (it contains domain words). It takes up one of your top-k retrieval slots and passes zero useful information to the LLM.

**Fix:** Filter by minimum character length after chunking:
```python
chunks = [chunk for chunk in chunks if len(chunk) > 50]
```

Use length as a proxy for information content. Titles are short. Content chunks are long.

### 10. Joining Chunks Back Together

**Why `"\n\n".join(current_chunk)` not `" ".join()`:**
- Paragraphs in the original document were separated by `\n\n`
- Joining with `\n\n` preserves the original formatting
- The LLM reads the chunk as a coherent document section, not a wall of text

**Why `.append()` not `chunks = chunks.append()`:**
- `.append()` modifies the list in place and returns `None`
- `chunks = chunks.append(...)` sets `chunks = None` → destroys your list
- Always call `.append()` without assignment

## 💻 Code

In [ ]:
# Day 9 — Structure-Aware Chunker
import re

sample_hr_policy = """  HUMAN RESOURCES POLICY

1. Employee Conduct and Workplace Ethics

All employees are expected to maintain professional behavior, treat colleagues with respect, and uphold the company's values at all times. Harassment, discrimination, bullying, or any form of unethical conduct will not be tolerated. Employees must comply with all company policies and maintain confidentiality of company information.

2. Attendance, Working Hours, and Leave

Employees are required to adhere to their assigned work schedules and notify their reporting manager in advance of any planned absence whenever possible. Attendance records will be maintained, and repeated unexplained absences may result in disciplinary action. Leave requests must be submitted through the approved process and are subject to managerial approval based on business requirements.

3. Performance Management and Professional Development

Employee performance will be reviewed periodically based on individual goals, job responsibilities, and overall contribution to the organization. Constructive feedback, coaching, and development opportunities will be provided to support professional growth. Consistent underperformance may lead to a performance improvement plan and further corrective actions if necessary.
"""


def chunk_by_structure(text):
    pattern = r"^\d+\. "
    paragraphs = text.split('\n\n')
    chunks = []
    current_chunk = []

    for paragraph in paragraphs:
        if re.match(pattern, paragraph):
            # New section header found — finalize previous section
            if current_chunk:
                chunks.append("\n\n".join(current_chunk))
            # Start fresh with this header
            current_chunk = [paragraph]
        else:
            # Body text — belongs to current section
            current_chunk.append(paragraph)

    # Finalize last section after loop ends
    if current_chunk:
        chunks.append("\n\n".join(current_chunk))

    return chunks


# Run and filter title chunk
chunks = chunk_by_structure(sample_hr_policy)
chunks = [chunk for chunk in chunks if len(chunk) > 50]

print(f"Total chunks: {len(chunks)}")
print("=" * 50)
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:")
    print(chunk)
    print("-" * 50)

In [ ]:
# Day 9 — Semantic Chunker
from sentence_transformers import SentenceTransformer
import numpy as np
import nltk

model = SentenceTransformer("all-MiniLM-L6-v2")


def chunk_by_semantics(text, threshold=0.25):
    # Step 1: Split into sentences
    sentences = nltk.sent_tokenize(text)

    # Step 2: Filter fragments
    sentences = [s.strip() for s in sentences if len(s.strip()) > 30]

    # Step 3: Embed all sentences
    embeddings = model.encode(sentences)

    # Step 4: Compute consecutive cosine similarities and split
    chunks = []
    current_chunk = [sentences[0]]

    for i in range(len(sentences) - 1):
        vec1 = embeddings[i]
        vec2 = embeddings[i + 1]

        dot_product = np.dot(vec1, vec2)
        magnitude = np.linalg.norm(vec1) * np.linalg.norm(vec2)
        similarity = dot_product / magnitude

        if similarity < threshold:
            # Topic boundary detected — finalize current chunk
            chunks.append("\n\n".join(current_chunk))
            current_chunk = [sentences[i + 1]]
        else:
            # Same topic — keep building current chunk
            current_chunk.append(sentences[i + 1])

    # Finalize last chunk
    if current_chunk:
        chunks.append("\n\n".join(current_chunk))

    return chunks


chunks = chunk_by_semantics(sample_hr_policy, threshold=0.25)

print(f"Total chunks: {len(chunks)}")
print("=" * 50)
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}:")
    print(chunk)
    print("-" * 50)

## 🔬 Observations

### Structure-Aware Results
| Chunk | Content | Quality |
|-------|---------|--------|
| Chunk 1 | Section 1 header + full body | ✅ Perfect |
| Chunk 2 | Section 2 header + full body | ✅ Perfect |
| Chunk 3 | Section 3 header + full body | ✅ Perfect |

No topic bleeding. Each chunk is a self-contained section. Title chunk correctly filtered.

### Semantic Chunker Results (threshold=0.25)
| Pair | Similarity | Split? | Correct? |
|------|-----------|--------|----------|
| (0,1) | 0.3973 | No | ✅ |
| (1,2) | 0.2562 | Yes | ⚠️ Both still ethics section |
| (2,3) | 0.4379 | No | ✅ |
| (3,4) | 0.5153 | No | ✅ |
| (4,5) | 0.2073 | Yes | ⚠️ Both attendance section |
| (5,6) | 0.2483 | Yes | ⚠️ Leave orphaned |
| (6,7) | 0.5069 | No | ✅ |
| (7,8) | 0.4079 | No | ✅ |

**Root cause of errors:** `nltk.sent_tokenize` joins section headers with first body sentence, creating mixed vectors that confuse similarity scores.

**Conclusion:** For this document type, structure-aware chunking is the correct choice.

## ✅ Day 9 Summary

```
✓ Why fixed chunking produces confident hallucinations in production
✓ Two failure modes: topic bleeding and loss of context
✓ Three chunking families: fixed, structure-aware, semantic
✓ Natural unit concept — what should never be split in each document type
✓ Structure-aware chunking algorithm — regex pattern matching on \n\n and ^\d+\.
✓ Two-list pattern: chunks (final) and current_chunk (temporary)
✓ Finalization happens in exactly two places: new header + after loop
✓ Title chunk filtering by character length
✓ Semantic chunking algorithm — consecutive cosine similarity
✓ Why nltk.sent_tokenize over splitting on period
✓ Fragment filtering with length threshold
✓ n-1 consecutive pairs for n sentences
✓ Threshold tuning: higher = more splits, lower = fewer splits
✓ Semantic chunking limitation on structured documents
✓ Production routing — chunking router pattern
✓ LLM-guided document routing using Day 3 structured output
✓ Adaptive chunking — inspect structure during chunking pass
✓ Hierarchical chunking — structure-aware + semantic combined
✓ append() vs chunks = chunks.append() — why assignment destroys the list
```

## ⚠️ Common Mistakes to Avoid

1. **`chunks = chunks.append(...)`** — `.append()` returns `None`. Never assign it.
2. **Finalizing inside the loop** — finalize only when hitting a new header or after loop ends, never on every iteration.
3. **Splitting on `.`** — use `nltk.sent_tokenize` to handle numbered lists, abbreviations, decimals.
4. **Hardcoding document variable inside function** — always use the `text` parameter, not a global variable.
5. **Using `range(sentences)`** — `range()` needs an integer, not a list. Use `range(len(sentences) - 1)`.
6. **`embeddings[sentence - 1]` when sentence=0`** — gives `embeddings[-1]` (last element). Use `sentence + 1` for consecutive pairs.
7. **Not adding next sentence to current_chunk after split** — sentence gets lost. After `current_chunk = []`, immediately append `sentences[i+1]`.

## 🔜 Coming in Day 10 — Advanced Retrieval

You now have clean chunks from structured documents. The next problem:
- Simple top-k retrieval returns the most similar chunks — but similarity alone is not enough
- Two chunks can be highly similar but say the same thing (redundancy)
- A query can have multiple sub-questions requiring chunks from different sections

**Day 10 covers:**
- MMR (Maximal Marginal Relevance) — relevance + diversity combined
- Hybrid Search — keyword + semantic combined
- Re-ranking — retrieve more, then re-score with a better model

**Pre-Day 10 question:**  
If you retrieve top-3 chunks and all three say roughly the same thing, what problem does that create for the LLM's answer quality — and how would you fix it?